# Splitters: TimeSplitter and BySourceSplitter

This tutorial introduces two data splitters for train/val/test partitioning:

- **TimeSplitter** — Splits a single time series by temporal order (with lookback overlap).
- **BySourceSplitter** — Splits multi-source data by assigning each source to a split.

## Splitter Input → Output

```mermaid
flowchart LR
    subgraph TimeSplitter
        T1["data: np.ndarray\n(500, 3)"] --> T2[get_splits / split_data]
        T2 --> T3["splits_dict: train/val/test\n(start, end) tuples"]
        T2 --> T4["splitted_data: arrays per split"]
    end
    subgraph BySourceSplitter
        B1["data_list: [dict_a, dict_b, dict_c]\nsource_names: [a,b,c]"] --> B2[split_data]
        B2 --> B3["result[key][split]: train/val/test\nper data key"]
    end
```

## When to Use Which

| Use Case | Splitter |
|----------|----------|
| Single time series (e.g. one sensor, one unit) | **TimeSplitter** — temporal order matters; train on past, validate/test on future |
| Multiple units/sources (e.g. units a, b, c) | **BySourceSplitter** — assign units to splits; train on unit a, validate on b, test on c (cross-unit generalization) |

## TimeSplitter

`TimeSplitter` splits a single array by time. Use `train`/`val`/`test` as float ratios (with `test=None` to derive test from remainder) or as integer sample counts. `seq_len`, `label_len`, `pred_len` control lookback overlap between splits.

In [ ]:
import numpy as np
from picid.data.split_strategies import TimeSplitter

data = np.random.randn(500, 3).astype(np.float32)
splitter = TimeSplitter(
    train=0.6,
    val=0.2,
    test=None,  # test=None: remainder becomes test
    seq_len=4,
    label_len=2,
    pred_len=2,
)
splits_dict, masks = splitter.get_splits(data)
print(
    f"Splits: train {splits_dict['train']}, val {splits_dict['val']}, test {splits_dict['test']}"
)

In [ ]:
splitted_data, split_masks = splitter.split_data({"features": data}, "features")
print(f"Train shape: {splitted_data['train'].shape}")
print(f"Val shape: {splitted_data['val'].shape}")
print(f"Test shape: {splitted_data['test'].shape}")

## BySourceSplitter

`BySourceSplitter` assigns each source (unit) to a split. Pass a list of dict-like data containers and a matching list of source names.

In [ ]:
from picid.data.split_strategies import BySourceSplitter

arr_a = np.random.randn(10, 3).astype(np.float32)
arr_b = np.random.randn(10, 3).astype(np.float32)
arr_c = np.random.randn(10, 3).astype(np.float32)
data_list = [
    {"features": arr_a},
    {"features": arr_b},
    {"features": arr_c},
]
source_names = ["src1", "src2", "src3"]
splitter2 = BySourceSplitter(
    sources_train=["src1"],
    sources_val=["src2"],
    sources_test=["src3"],
)
result = splitter2.split_data(data_list, source_names)
print(f"Train: {result['features']['train']}")
print(f"Val: {result['features']['val']}")
print(f"Test: {result['features']['test']}")

## Summary

- **TimeSplitter**: `get_splits(data)` → (splits_dict, masks); `split_data(data_dict, split_variable)` → (splitted_data, masks). Use for single time series.
- **BySourceSplitter**: `split_data(data_list, source_name_lst)` → result with `result[key][split]` per data key. Use for multi-source / cross-unit evaluation.